# 01 — Prepare Dataset items_prompts_tv_2

**Mục đích:** Build prompt/completion từ `SeanSunny/items_tv_v6` và push lên HuggingFace  
thành `SeanSunny/items_prompts_tv_2` (3 splits: train/val/test).

**Filter:** train/val lọc `price <= 1,000,000 VND`. Test giữ nguyên.  
**Completion:** `round(price/1000)` cho cả 3 splits.  
**Schema:** `prompt`, `completion`, `price_vnd_true` (không có `category`).

**Không cần GPU.**

**Yêu cầu:** `HF_TOKEN` có write access vào HuggingFace.

**Output:** `SeanSunny/items_prompts_tv_2` trên HF, `dataset_stats.md`

In [ ]:
# On Colab: uncomment and run this cell first
# !pip install datasets huggingface_hub --quiet

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import load_dataset, DatasetDict

# ---- Config ----
SOURCE_DATASET = "SeanSunny/items_tv_v6"
OUTPUT_DATASET = "SeanSunny/items_prompts_tv_2"
SEED = 42

# HF_TOKEN: set env variable or paste below
HF_TOKEN = os.environ.get("HF_TOKEN", "")
# HF_TOKEN = "hf_xxx"  # uncomment if not in env

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not set. Export it or set HF_TOKEN above.")

random.seed(SEED)
np.random.seed(SEED)
print("Config OK")

In [2]:
# Load source dataset
ds = load_dataset(SOURCE_DATASET)
print(ds)

splits = list(ds.keys())
val_key = "val" if "val" in splits else "validation"
print(f"\nAvailable splits: {splits}")
print(f"Using val split  : '{val_key}'")
print(f"Columns          : {ds['train'].column_names}")

DatasetDict({
    train: Dataset({
        features: ['title', 'category', 'price', 'full', 'brand', 'summary', 'prompt', 'id'],
        num_rows: 110000
    })
    validation: Dataset({
        features: ['title', 'category', 'price', 'full', 'brand', 'summary', 'prompt', 'id'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['title', 'category', 'price', 'full', 'brand', 'summary', 'prompt', 'id'],
        num_rows: 5000
    })
})

Available splits: ['train', 'validation', 'test']
Using val split  : 'validation'
Columns          : ['title', 'category', 'price', 'full', 'brand', 'summary', 'prompt', 'id']


In [3]:
MAX_PRICE = 1_000_000  # VND — consistent with Day 3/4

def filter_by_price(split, max_price=MAX_PRICE):
    return split.filter(lambda item: float(item["price"]) <= max_price)

n_train_before = len(ds["train"])
n_val_before   = len(ds[val_key])

ds_train_filtered = filter_by_price(ds["train"])
ds_val_filtered   = filter_by_price(ds[val_key])
ds_test_raw       = ds["test"]  # no filter — keep original test for fair comparison with v8

print(f"Train: {n_train_before:,} -> {len(ds_train_filtered):,} items after price filter "
      f"(dropped {n_train_before - len(ds_train_filtered):,}, max_price={MAX_PRICE:,})")
print(f"Val  : {n_val_before:,} -> {len(ds_val_filtered):,} items after price filter "
      f"(dropped {n_val_before - len(ds_val_filtered):,})")
print(f"Test : {len(ds_test_raw):,} -> {len(ds_test_raw):,} items (no filter applied)")

Filter:   0%|          | 0/110000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

Train: 110,000 -> 85,727 items after price filter (dropped 24,273, max_price=1,000,000)
Val  : 5,000 -> 3,926 items after price filter (dropped 1,074)
Test : 5,000 -> 5,000 items (no filter applied)


In [ ]:
# Prompt builder — uses 'summary' column (pre-formatted, always non-null)
# summary already contains: Tiêu đề / Danh mục / Thương hiệu / Mô tả / Thông số

def build_prompt(item: dict) -> str:
    summary = (item.get("summary") or "").strip()
    return f"Sản phẩm này có giá bao nhiêu ?\n{summary}\n\nGiá là: "


def build_example(item: dict) -> dict:
    price = float(item["price"])
    return {
        "prompt": build_prompt(item),
        "completion": str(int(round(price / 1000))),
        "price_vnd_true": int(round(price)),
    }


# Sanity check
item0 = dict(ds_train_filtered[0])
ex0 = build_example(item0)
print("=== Sample train example ===")
print(ex0["prompt"])
print(f"completion     : {ex0['completion']}")
print(f"price_vnd_true : {ex0['price_vnd_true']:,}")

In [ ]:
# Apply to all splits — same completion format (round(price/1000)) for all 3 splits
def process_split(split):
    return split.map(
        build_example,
        remove_columns=split.column_names,
        desc="Building prompts",
        num_proc=1,
    )

train_prompts = process_split(ds_train_filtered)
val_prompts   = process_split(ds_val_filtered)
test_prompts  = process_split(ds_test_raw)

out_ds = DatasetDict({
    "train": train_prompts,
    "val":   val_prompts,
    "test":  test_prompts,
})

print(out_ds)
print(f"\nSchema: {out_ds['train'].column_names}")

In [ ]:
# Dataset statistics
for split_name, split_data in out_ds.items():
    df = split_data.to_pandas()
    print(f"\n=== {split_name} ({len(df):,} items) ===")
    prices = df["price_vnd_true"]
    print(f"  Price (VND): mean={prices.mean():.0f}, median={prices.median():.0f}, "
          f"min={prices.min():,}, max={prices.max():,}")
    print(f"  Completion sample: {df['completion'].sample(5, random_state=42).tolist()}")

In [7]:
# Show 5 complete examples from train
df_train = out_ds["train"].to_pandas()
sample5 = df_train.sample(5, random_state=42)

for _, row in sample5.iterrows():
    print("=" * 70)
    print(row["prompt"])
    print(f"[COMPLETION: {row['completion']}  |  PRICE_VND: {row['price_vnd_true']:,}]")
    print()

Sản phẩm này có giá bao nhiêu ?
Tiêu đề: Ổ khóa đĩa hợp kim siêu chịu lực  
Danh mục: Phụ kiện xe máy  
Thương hiệu: PaKaSa  
Mô tả: Ổ khóa đĩa chống trộm, cốt inox 100% và chất liệu hợp kim siêu chịu lực, kèm 2 chìa khóa.  
Thông số: Khả năng chịu lực cao, không bị gỉ, cấu trúc ruột bi, bảo hành 6 tháng.

Giá là: 
[COMPLETION: 60  |  PRICE_VND: 59,590]

Sản phẩm này có giá bao nhiêu ?
Tiêu đề: Ốp Hộp Ốp Lưng LAUT HUEX SLIM iPhone 16 Pro Max  
Danh mục: Phụ kiện điện thoại  
Thương hiệu: LAUT  
Mô tả: Ốp lưng siêu mỏng, làm từ nhựa tái chế 100% với lớp lót microfiber, chống va đập và trượt.  
Thông số: Hỗ trợ sạc không dây Qi, chịu rơi tới 2m, bảo vệ camera và chống trầy xước.

Giá là: 
[COMPLETION: 690  |  PRICE_VND: 690,000]

Sản phẩm này có giá bao nhiêu ?
Tiêu đề: Thắt lưng nam vải bố kiểu lính US ARMY  
Danh mục: Phụ kiện nam  
Thương hiệu: US ARMY  
Mô tả: Thắt lưng vải bố chống gỉ, kiểu dáng lính với mặt khóa hợp kim thép cao cấp, phù hợp cho mọi phong cách.  
Thông số: Độ bền c

In [ ]:
# Save dataset_stats.md
lines = ["# Dataset Stats — items_prompts_tv_2\n\n"]

for split_name in ["train", "val", "test"]:
    df = out_ds[split_name].to_pandas()
    prices = df["price_vnd_true"]
    lines.append(f"## {split_name} ({len(df):,} items)\n\n")
    lines.append(f"- Price (VND): mean={prices.mean():.0f}, median={prices.median():.0f}, "
                 f"min={prices.min():,}, max={prices.max():,}\n")
    lines.append("\n")

lines.append("## Sample 5 Prompts (train)\n\n")
df_train = out_ds["train"].to_pandas()
for _, row in df_train.sample(5, random_state=42).iterrows():
    lines.append("```\n")
    lines.append(row["prompt"].rstrip() + "\n")
    lines.append(f"[COMPLETION: {row['completion']}  |  PRICE_VND: {row['price_vnd_true']:,}]\n")
    lines.append("```\n\n")

with open("dataset_stats.md", "w", encoding="utf-8") as f:
    f.writelines(lines)

print("Saved dataset_stats.md")

## Ket qua Phase 0 Notebook 2

**DONE:** Dataset `SeanSunny/items_prompts_tv_2` da push len HuggingFace.  
Share `dataset_stats.md` voi Claude de confirm truoc khi sang Phase 1.

**Buoc tiep:**
1. Share `profile_results.json` (tu notebook 00) voi Claude
2. Share `dataset_stats.md` voi Claude
3. Claude confirm `max_seq_length` va `max_new_tokens`
4. Bat dau Phase 1: `02_baseline_v0.ipynb` (can GPU)